# 04 — Machine-learning models (Phase 7)

Classifiers are fit in **time order**. The scaler for logistic regression is fit on the **train** fold only. The model with the best **validation** ROC-AUC is selected; the test fold is a report, not a tuning knob.

This notebook does not claim an edge. The bundled sample is synthetic.

```bash
python -m quant_forex ingest --config configs/default.yaml
python -m quant_forex train --config configs/default.yaml
```


In [ ]:
from quant_forex.config.loader import load_config
from quant_forex.exploration.io import latest_processed_csv, load_processed_ohlcv
from quant_forex.features.engine import build_features
from quant_forex.models.pipeline import run_ml_pipeline
from quant_forex.models.splits import chronological_masks
from quant_forex.models.target import TARGET_COL, add_next_return_target, feature_matrix_columns
from quant_forex.utils.paths import ProjectPaths

paths = ProjectPaths.find()
config = load_config(paths.resolve("configs/default.yaml"))
processed = latest_processed_csv(paths.data_processed)
ohlcv = load_processed_ohlcv(processed)
built = build_features(ohlcv, config.features)
labeled = add_next_return_target(built.frame, config.models)
cols = feature_matrix_columns(labeled)
assert TARGET_COL not in cols
print("features", len(cols), "target", TARGET_COL)
labeled[["timestamp", "close", TARGET_COL]].tail()

In [ ]:
usable = labeled.dropna(subset=cols + [TARGET_COL]).reset_index(drop=True)
train, val, test, method = chronological_masks(usable["timestamp"], config.splits)
print(method, int(train.sum()), int(val.sum()), int(test.sum()))
print("train max", usable.loc[train, "timestamp"].max())
print("val min", usable.loc[val, "timestamp"].min())
print("test min", usable.loc[test, "timestamp"].min())

In [ ]:
result = run_ml_pipeline(config, paths, input_csv=processed)
print("selected", result.selected_model)
print(result.report_path)
print(result.predictions_path)
print(result.artifact_path)
if result.signals:
    print(result.signals[0])